<a href="https://colab.research.google.com/github/YifeiCAO/CogModelingRNNsTutorial/blob/main/Copy_of_Human_dataset_inner_outer_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import and preparation

In [ ]:
AS_OUTER_LOOP = [1,2,3,4,5,6,7,8,9,10] # assigned outer loop(s) to run in this notebook

In [ ]:
!pip -q install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 102.6 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd
from io import StringIO
import gdown

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import numpy as np
import pandas as pd
# import plotnine as gg
# gg.theme_set(gg.theme_classic)  # for nicer-looking plots
import jax.numpy as jnp
import jax
import optax
import scipy

!pip install -U dm-haiku
import haiku as hk
rng_seq = hk.PRNGSequence(np.random.randint(2**32))

#@title Install required packages.
try:
    from google.colab import files
    _ON_COLAB = True
except:
    _ON_COLAB = False

if _ON_COLAB:
  !rm -rf CogModelingRNNsTutorial
  !git clone https://github.com/YifeiCAO/CogModelingRNNsTutorial
  !pip install -e CogModelingRNNsTutorial/CogModelingRNNsTutorial
  !cp CogModelingRNNsTutorial/CogModelingRNNsTutorial/*py CogModelingRNNsTutorial
else:
  !pip install CogModelingRNNsTutorial/requirements.txt

#@title Imports + defaults settings.
# %load_ext autoreload
# %autoreload 2
# for reload
# %reload_ext autoreload

# import haiku as hk
# import jax
# import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optax
import os
import warnings

# warnings.filterwarnings("ignore")

# try:
#     from google.colab import files
#     _ON_COLAB = True
# except:
#     _ON_COLAB = False

from CogModelingRNNsTutorial import bandits
from CogModelingRNNsTutorial import disrnn
from CogModelingRNNsTutorial import hybrnn
from CogModelingRNNsTutorial import hybconrnn
from CogModelingRNNsTutorial import hybrnn_direct_con
from CogModelingRNNsTutorial import plotting
from CogModelingRNNsTutorial import rat_data
from CogModelingRNNsTutorial import rnn_utils

from google.colab import drive
import pickle

# 挂载 Google Drive
drive.mount('/content/drive')


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 15.2 MB/s eta 0:00:00
Cloning into 'CogModelingRNNsTutorial'...
remote: Enumerating objects: 1379, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 1379 (delta 213), reused 192 (delta 161), pack-reused 1112 (from 1)
Receiving objects: 100% (1379/1379), 6.88 MiB | 43.79 MiB/s, done.
Resolving deltas: 100% (886/886), done.
Obtaining file:///content/CogModelingRNNsTutorial/CogModelingRNNsTutorial
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for CogModelingRNNsTutorial (pyproject.toml) ... done
  Created wheel for CogModelingRNNsTutorial: filename=cogmodelingrnnstutorial-0.0.0-0.editable-py3-none-any.whl size=3095 sha256=5a1263aaef7da85bae1ca4271730a451e0451f4c662096582129c2af3834d77

In [ ]:
def compute_negative_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]

    # 直接在外部运行 eval_model（不要放 jit 里！）
    model_outputs, _ = rnn_utils.eval_model(model_fun, params, xs)

    # 再 jit 一个小函数，只对 model_outputs 做 softmax
    @jax.jit
    def compute_log_probs(logits):
        return jax.nn.log_softmax(logits[:, :, :2], axis=-1)

    predicted_log_choice_probabilities = compute_log_probs(model_outputs)

    # 后续仍转为 NumPy 做索引
    predicted_log_choice_probabilities = np.array(predicted_log_choice_probabilities)

    total_log_likelihood = 0.0
    total_valid_trials = 0

    for sess_i in range(n_sessions):
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            if actual_choice >= 0:
                total_log_likelihood += predicted_log_choice_probabilities[trial_i, sess_i, actual_choice]
                total_valid_trials += 1

    if total_valid_trials == 0:
        raise ValueError("No valid trials found.")

    avg_nll = -total_log_likelihood / total_valid_trials
    return avg_nll



def evaluate_nll_over_full_dataset(dataset, model_fun, params):
    all_nlls = []
    for _ in range(dataset.n_batches):
        nll = compute_negative_log_likelihood(dataset, model_fun, params)
        all_nlls.append(nll)

    avg_nll = np.mean(all_nlls)
    print(f"[All Batches Averaged] NLL: {avg_nll:.4f}")
    return avg_nll


In [ ]:
@jax.jit
def compute_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]
    model_outputs, model_states = rnn_utils.eval_model(model_fun, params, xs)

    # predicted log-probs for the first two actions
    predicted_log_choice_probabilities = np.array(
        jax.nn.log_softmax(model_outputs[:, :, :2], axis=-1)
    )

    n_actions = predicted_log_choice_probabilities.shape[2]
    log_likelihoods = []

    for sess_i in range(n_sessions):
        log_likelihood = 0.0
        n = 0
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            # ignore invalid trials (<0 or ≥n_actions)
            if 0 <= actual_choice < n_actions:
                log_likelihood += predicted_log_choice_probabilities[
                    trial_i, sess_i, actual_choice
                ]
                n += 1

        if n > 0:
            normalized_likelihood = np.exp(log_likelihood / n)
            log_likelihoods.append(normalized_likelihood)

    mean_likelihood = np.mean(log_likelihoods)
    std_likelihood  = np.std(log_likelihoods)

    print(f'Average Normalized Likelihood: {100 * mean_likelihood:.1f}%')
    return mean_likelihood, std_likelihood


In [ ]:
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

## Human Datas

In [ ]:
osf_url = 'https://osf.io/download/xe6yu/'
response = requests.get(osf_url)

# Check the request
if response.status_code == 200:
    # Read as pandas dataframe
    qasim_data = pd.read_csv(StringIO(response.text))
    print('File downloaded and read successfully!')
else:
    print('Failed to download file. Status code:', response.status_code)

qasim_data.head()

# # read data
selected_columns = ['participant', 'trials_gamble', 'gamble', 'prob', 'reward']

qasim = qasim_data[selected_columns]
qasim_filtered = qasim[qasim['trials_gamble'].notna()]
qasim_sorted = qasim_filtered.groupby('participant', group_keys=False).apply(lambda x: x.sort_values('trials_gamble'))
qasim_sorted = qasim_sorted.reset_index(drop=True)
qasim_sorted['participant'] = qasim_sorted.groupby(['participant']).ngroup() + 1
qasim_sorted['action'] = qasim_sorted['gamble']
qasim_sorted

# —— 2) 把缺失的 action 填成 -1 —— #
qasim_sorted['action'] = qasim_sorted['action'].fillna(-1).astype(int)

# —— 3) 如果需要，把 reward 映射到 0/1 —— #
# （如果已经是 0/1 可跳过；否则取消下面注释并调整映射字典）
# qasim_sorted['reward'] = (
#     qasim_sorted['reward']
#     .map({-1: 0, 1: 1})
#     .fillna(-1)
#     .astype(int)
# )

# —— 4) 排序，确保 trial 顺序 —— #
qasim_sorted = qasim_sorted.sort_values(
    ['participant', 'trials_gamble']
).reset_index(drop=True)

# —— 5) 生成下一步动作 action_n —— #
qasim_sorted['action_n'] = (
    qasim_sorted
    .groupby('participant')['action']
    .shift(-1)
)
# 每个 participant 最后一 trial 的 action_n 设为 -1
last_idxs = qasim_sorted.groupby('participant').tail(1).index
qasim_sorted.loc[last_idxs, 'action_n'] = -1

# —— 6) 按 participant 构造 xs_list, ys_list —— #
xs_list, ys_list = [], []
for pid, grp in qasim_sorted.groupby('participant'):
    grp = grp.sort_values('trials_gamble')
    x = grp[['action', 'reward']].to_numpy().astype(float)    # 输入特征
    y = grp[['action_n']].to_numpy().astype(int)             # 下一步动作
    xs_list.append(x)
    ys_list.append(y)

# —— 7) 堆成三维数组 —— #
xs_qa = np.stack(xs_list, axis=1)  # (n_sessions, n_trials, 2)
ys_qa = np.stack(ys_list, axis=1)  # (n_sessions, n_trials, 1)

print("xs.shape =", xs_qa.shape)
print("ys.shape =", ys_qa.shape)



File downloaded and read successfully!
xs.shape = (60, 206, 2)
ys.shape = (60, 206, 1)


### Sidarus dataset

In [ ]:
# # 修改后的 file_id
# file_id = '1TSV6CdyClKz831qD2ln4z3WjLLcOgG1n'
# download_url = f'https://drive.google.com/uc?id={file_id}'

# # 下载并保存为 'downloaded_file.csv'
# output_file = 'downloaded_file.csv'
# gdown.download(download_url, output_file, quiet=False)

# # 读取 CSV 数据
# sida_data = pd.read_csv(output_file)
# sida_data

# # 1) action 从 {1,2} → {0,1}，并把所有缺失值填成 -1
# sida_data['action'] = (
#     sida_data['action']
#     .map({1: 0, 2: 1})    # 映射
#     .fillna(-1)           # 缺失值设为 -1
#     .astype(int)
# )

# # 2) outcome 从 {-1,1} → {0,1}，缺失也填成 -1
# sida_data['reward'] = (
#     sida_data['outcome']
#     .map({-1: 0, 1: 1})
#     .fillna(-1)
#     .astype(int)
# )

# # 3) 给每个被试×session 分配一个连续的 session_id
# sida_data['session_id'] = (
#     sida_data
#     .groupby(['subj', 'session'], sort=False)
#     .ngroup()
#     + 1
# )

# # 4) 排序，保证 trial 顺序
# sida_data = sida_data.sort_values(
#     ['session_id', 'epN', 'epTrialN']
# ).reset_index(drop=True)

# # 5) 生成下一步动作 action_n；每个 session 末尾设为 -1
# sida_data['action_n'] = (
#     sida_data
#     .groupby('session_id')['action']
#     .shift(-1)
# )
# last_idx = sida_data.groupby('session_id').tail(1).index
# sida_data.loc[last_idx, 'action_n'] = -1

# # 6) 按 session_id 抽取序列，并堆成 xs, ys
# xs_list, ys_list = [], []
# for sid, grp in sida_data.groupby('session_id'):
#     grp = grp.sort_values(['epN', 'epTrialN'])
#     x = grp[['action', 'reward']].to_numpy().astype(float)  # 特征
#     y = grp[['action_n']].to_numpy().astype(int)              # Label
#     xs_list.append(x)
#     ys_list.append(y)

# # 最终结果：xs.shape == (n_sessions, n_trials, 2)，ys.shape == (n_sessions, n_trials, 1)
# xs_sida = np.stack(xs_list, axis=1)
# ys_sida = np.stack(ys_list, axis=1)

# print("xs.shape =", xs_sida.shape)
# print("ys.shape =", ys_sida.shape)


### Schaaf dataset

In [ ]:
# 修改后的 file_id
file_id = '1rJFmDhCE3fdXtSHSSvtre_7V49gGsg7P'
download_url = f'https://drive.google.com/uc?id={file_id}'

# 下载并保存为 'downloaded_file.csv'
output_file = 'downloaded_file.csv'
gdown.download(download_url, output_file, quiet=False)

# 读取 CSV 数据
schaaf_data = pd.read_csv(output_file)
schaaf_data

df1 = schaaf_data[['pp', 'trial1', 'response1', 'outcome1']].copy()
df1.columns = ['pp', 'trial_in_session', 'response', 'reward']
df1['session'] = 1

df2 = schaaf_data[['pp', 'trial2', 'response2', 'outcome2']].copy()
df2.columns = ['pp', 'trial_in_session', 'response', 'reward']
df2['session'] = 2

# —— 上面拆分、concat、sort 的部分保持不变 —— #

# 合并成长表
df_long = pd.concat([df1, df2], ignore_index=True)
df_long = df_long.sort_values(['pp', 'session', 'trial_in_session']).reset_index(drop=True)

# —— 映射 outcome，再填 missing response —— #
# 把原来的 response1/response2 改名后是 `response`
# 把 outcome1/outcome2 改名后是 `reward`
# 把 outcome 映射成 1/0，然后把原本缺失的 reward 补成 -1
df_long['reward'] = (
    df_long['reward']
    .map({1: 1, -1: 0})
    .fillna(-1)
    .astype(int)
)

# 把缺失的 response 一样补成 -1
df_long['response'] = (
    df_long['response']
    .fillna(-1)
    .astype(int)
)


# 接下来再做 session_id、shift action_n、以及 stack xs/ys 的流程……
df_long['session_id'] = (df_long['pp'] - 1) * 2 + df_long['session']
df_long['action_n'] = df_long.groupby('session_id')['response'].shift(-1)
last_idx = df_long.groupby('session_id').tail(1).index
df_long.loc[last_idx, 'action_n'] = -1

# 生成 xs, ys
session_ids = df_long['session_id'].unique()
xs_list, ys_list = [], []
for sid in session_ids:
    sd = df_long[df_long['session_id'] == sid].sort_values('trial_in_session')
    x = sd[['response', 'reward']].to_numpy().astype(float)
    y = sd[['action_n']].to_numpy().astype(int)
    xs_list.append(x)
    ys_list.append(y)

xs = np.stack(xs_list, axis=0)   # (n_sessions, n_trials, 2)
ys = np.stack(ys_list, axis=0)   # (n_sessions, n_trials, 1)


# 或者第 0 维是 trials，第 1 维是 sessions：
xs_sch = np.stack(xs_list, axis=1)  # (n_trials, n_sessions, 2)
ys_sch = np.stack(ys_list, axis=1)  # (n_trials, n_sessions, 1)

print("xs.shape =", xs_sch.shape)
print("ys.shape =", ys_sch.shape)

# 1 → 0
# 方法一：一次性用 np.where 嵌套
xs_sch[:, :, 0] = np.where(
    xs_sch[:, :, 0] == 2,  # 如果等于 2
    1,                         # 则变成 1
    np.where(
        xs_sch[:, :, 0] == 1,  # 否则如果等于 1
        0,                         # 则变成 0
        xs_sch[:, :, 0]        # 其它值保持原样
    )
)
ys_sch[ys_sch == 1] = 0
ys_sch[ys_sch == 2] = 1

Downloading...
From: https://drive.google.com/uc?id=1rJFmDhCE3fdXtSHSSvtre_7V49gGsg7P
To: /content/downloaded_file.csv
100%|██████████| 374k/374k [00:00<00:00, 152MB/s]

xs.shape = (249, 94, 2)
ys.shape = (249, 94, 1)


In [ ]:
# xs_flat = xs_sch[:-1, :, 1]  # 或用 ys[:,:,0]
# # 1) 有效 trial 的掩码（0 或 1 才算），shape (n_trials, n_sessions)
# valid = xs_flat >= 0
# # 2) 每个 session 成功的次数（sum of 1s）
# n_success = np.sum(xs_flat * valid, axis=0)
# # 3) 每个 session 的有效 trial 数
# n_valid   = np.sum(valid, axis=0)
# # 4) 正确率 = 成功次数 / 有效数
# accuracy_per_session = n_success / n_valid

# # 打印
# for i, acc in enumerate(accuracy_per_session):
#     print(f"Session {i:02d}: accuracy = {acc:.4f} ({acc*100:.2f}%)")

### Maria Dataset

In [ ]:
import pandas as pd
import numpy as np
import gdown

# —— 1) 下载并读入原始数据 —— #
file_id = '1N_zAy-qrbfjvF8Kbb504IH2JNhR5KI-P'
url     = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, 'downloaded_file.csv', quiet=False)
eck_data = pd.read_csv('downloaded_file.csv')

# —— 2) 筛选＆重命名列 —— #
sel = ['sID','TrialID','selected_box','reward']
eck_sorted = eck_data[sel].copy()
eck_sorted['participant'] = eck_sorted.groupby('sID').ngroup()+1
eck_sorted['action']      = eck_sorted['selected_box']

# 只保留至少做够 120 试次的被试
max_trial = eck_sorted.groupby('participant')['TrialID'].transform('max')
eck_sorted = eck_sorted[max_trial>=120]

# 只用前 120 试次
eck_sorted = eck_sorted[eck_sorted['TrialID']<=120].reset_index(drop=True)

# —— 3) 生成下一步动作 action_n —— #
def generate_action_n(group):
    group = group.sort_values('TrialID')
    group['action_n'] = group['action'].shift(-1).fillna(-1).astype(int)
    return group

eck_sorted = (
    eck_sorted
    .groupby('participant', group_keys=False)
    .apply(generate_action_n)
    .reset_index(drop=True)
)

# —— 4) 按 participant 构造 xs_list/ys_list —— #
xs_list, ys_list = [], []
for pid, grp in eck_sorted.groupby('participant'):
    grp = grp.sort_values('TrialID').iloc[:120]
    x = grp[['action','reward']].to_numpy().astype(float)  # (120,2)
    y = grp[['action_n']].to_numpy().astype(int)           # (120,1)
    xs_list.append(x)
    ys_list.append(y)

# —— 5) stack 成 (n_sessions, n_trials, feat_dim) —— #
xs_ma = np.stack(xs_list, axis=1)  # (305,120,2)
ys_ma = np.stack(ys_list, axis=1)  # (305,120,1)

print("xs_ma.shape =", xs_ma.shape)
print("ys_ma.shape =", ys_ma.shape)


Downloading...
From: https://drive.google.com/uc?id=1N_zAy-qrbfjvF8Kbb504IH2JNhR5KI-P
To: /content/downloaded_file.csv
100%|██████████| 2.57M/2.57M [00:00<00:00, 361MB/s]


xs_ma.shape = (120, 305, 2)
ys_ma.shape = (120, 305, 1)


## Clean Data

## Generate a big human dataset with all experiments, session length is 130 trial per session

In [ ]:
import numpy as np

def segment_and_pad(x: np.ndarray,
                    y: np.ndarray,
                    seg_len: int = 130,
                    pad_x: float = 0.,
                    pad_y: int = -1):
    T, D = x.shape
    n_segs = int(np.ceil(T / seg_len))
    x_segs, y_segs = [], []
    for i in range(n_segs):
        start = i * seg_len
        end = start + seg_len
        x_part = x[start : min(end, T)]
        y_part = y[start : min(end, T)]
        pad = end - min(end, T)
        if pad > 0:
            x_part = np.pad(x_part,
                            pad_width=((0, pad), (0, 0)),
                            constant_values=pad_x)
            y_part = np.pad(y_part,
                            pad_width=((0, pad), (0, 0)),
                            constant_values=pad_y)
        x_segs.append(x_part)
        y_segs.append(y_part)
    return x_segs, y_segs

# —— 假设你已经有这几组 (xs, ys) —— #
# xs_qa   (60,  206, 2), ys_qa  (60,206, 1)
# xs_sida (800, 40,  2), ys_sida(800,40, 1)
# xs_sch  (249, 44,  2), ys_sch (249,44, 1)
# xs_ma   (120,305, 2), ys_ma  (120,305,1)

all_xs, all_ys = [], []

for xs, ys in [(xs_qa, ys_qa),
               (xs_sch, ys_sch),
               (xs_ma, ys_ma)]:

    # 如果是 (n_trials, n_sessions, feat) 维度，就直接：
    # T, N, D = xs.shape
    # 否则若是 (n_sessions, n_trials, feat)，先转：
    # xs = xs.transpose(1,0,2)
    # ys = ys.transpose(1,0,2)

    T, N, D = xs.shape
    for sess in range(N):
        x_seq = xs[:, sess, :]  # (T, D)
        y_seq = ys[:, sess, :]  # (T, 1)
        x_segs, y_segs = segment_and_pad(
            x_seq, y_seq,
            seg_len=130,
            pad_x=0., pad_y=-1
        )
        all_xs.extend(x_segs)
        all_ys.extend(y_segs)

# —— 修改在这里：不要 np.stack，直接输出列表 —— #
# all_xs 是一个 Python list，长度 = 总片段数，每个元素 shape=(130, D)
# all_ys 是一个 Python list，长度 = 总片段数，每个元素 shape=(130, 1)

print(f"Total segments: {len(all_xs)}")
print(f"First xs segment shape: {all_xs[0].shape}")
print(f"First ys segment shape: {all_ys[0].shape}")

# 如果你需要把它们返回成变量：
xs_segment_list = all_xs
ys_segment_list = all_ys


Total segments: 699
First xs segment shape: (130, 2)
First ys segment shape: (130, 1)


In [ ]:
def format_into_datasets_10_multi(
    xs_list: list[np.ndarray],
    ys_list: list[np.ndarray],
    dataset_constructor,
    batch_size: int = None,
    random_seed: int = None,
):
    """
    对多只猴子独立做 10‐fold CV，每折：
      - test = 每只猴子第 i 份 fold
      - validation 抽取与 test 大小一致的 session
      - 剩余做 train
    Args:
      xs_list: list of np.ndarray, 每项形状 (timesteps, n_sess_i, feat)
      ys_list: list of np.ndarray, 每项形状 (timesteps, n_sess_i, ...)
      dataset_constructor: 接受 (xs_sub, ys_sub, batch_size) 的构造器
      batch_size: mini‐batch 大小；None 则每 dataset 用全部 episodes
      random_seed: int，可复现打乱
    Returns:
      List of 10 tuples (ds_train, ds_val, ds_test)
    """
    rng = np.random.RandomState(random_seed) if random_seed is not None else np.random

    n_monkeys = len(xs_list)
    # 1. 对每只猴子生成打乱后的 session 索引，并切成 10 份
    folds_per_monkey = []
    for xs in xs_list:
        n_sess = xs.shape[1]
        idx = rng.permutation(n_sess)
        folds = np.array_split(idx, 10)
        folds_per_monkey.append(folds)

    folds_datasets = []
    for i in range(10):
        # 2a. 测试集：三猴子各自的第 i 份
        test_parts = [folds_per_monkey[m][i] for m in range(n_monkeys)]

        # 2b. 剩余 sessions 用于 train+val
        rem_parts = [
            np.hstack([folds_per_monkey[m][j] for j in range(10) if j != i])
            for m in range(n_monkeys)
        ]

        # 2c. 验证集大小与测试集一致：每只猴子抽取与 test_parts[m] 相同数量
        val_parts = []
        train_parts = []
        for m in range(n_monkeys):
            rng.shuffle(rem_parts[m])
            n_test = len(test_parts[m])
            val = rem_parts[m][:n_test]
            train = rem_parts[m][n_test:]
            val_parts.append(val)
            train_parts.append(train)

        # 3. 拼接各猴子的 train/val/test xs, ys
        def _gather(parts_idx):
            xs_sub, ys_sub = [], []
            for m, idxs in enumerate(parts_idx):
                xs_sub.append(xs_list[m][:, idxs, :])
                ys_sub.append(ys_list[m][:, idxs, :])
            return np.concatenate(xs_sub, axis=1), np.concatenate(ys_sub, axis=1)

        xs_train, ys_train = _gather(train_parts)
        xs_val,   ys_val   = _gather(val_parts)
        xs_test,  ys_test  = _gather(test_parts)

        # 4. 构造 DatasetRNN
        ds_train = dataset_constructor(xs_train, ys_train, batch_size=batch_size)
        ds_val   = dataset_constructor(xs_val,   ys_val,   batch_size=batch_size)
        ds_test  = dataset_constructor(xs_test,  ys_test,  batch_size=batch_size)

        folds_datasets.append((ds_train, ds_val, ds_test))

    return folds_datasets


In [ ]:
# 假设你已经有原始的：
#   xs_qa   (T_qa,   N_qa,   D),   ys_qa   (T_qa,   N_qa,   1)
#   xs_sida (T_sida, N_sida, D),   ys_sida (T_sida, N_sida, 1)
#   xs_sch  (T_sch,  N_sch,  D),   ys_sch  (T_sch,  N_sch,  1)
#   xs_ma   (T_ma,   N_ma,   D),   ys_ma   (T_ma,   N_ma,   1)

def make_segmented_array(xs, ys, seg_len=130, pad_x=0., pad_y=-1):
    all_xs, all_ys = [], []
    T, N, D = xs.shape
    for sess in range(N):
        x_seq = xs[:, sess, :]    # (T, D)
        y_seq = ys[:, sess, :]    # (T, 1)
        x_segs, y_segs = segment_and_pad(x_seq, y_seq, seg_len, pad_x, pad_y)
        all_xs.extend(x_segs)     # list of (130, D)
        all_ys.extend(y_segs)     # list of (130, 1)
    # 把 list 再拼成一个三维 array (130, n_segments, D)
    xs_seg = np.stack(all_xs, axis=1)
    ys_seg = np.stack(all_ys, axis=1)
    return xs_seg, ys_seg

# 针对四个源分别做一次
xs_qa_seg,   ys_qa_seg   = make_segmented_array(xs_qa,   ys_qa)
xs_sch_seg, ys_sch_seg  = make_segmented_array(xs_sch,  ys_sch)
xs_ma_seg,  ys_ma_seg   = make_segmented_array(xs_ma,   ys_ma)

In [ ]:
xs_human_list = [xs_qa_seg, xs_sch_seg, xs_ma_seg]
ys_human_list = [ys_qa_seg, ys_sch_seg, ys_ma_seg]

# Run cross validation on different models

## Vanilla RNN inner-outer loop

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_vrnn_inner_search_results_sub3.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_vrnn_outer_results_sub3.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx :', test_idx)

    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            def make_rnn():
                return hk.DeepRNN([hk.VanillaRNN(hs), hk.Linear(output_size=2)])

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_rnn,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_rnn, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    def make_best_rnn():
        return hk.DeepRNN([hk.VanillaRNN(best_params['hidden_size']), hk.Linear(output_size=2)])

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_rnn,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_rnn, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


Streaming output truncated to the last 5000 lines.
updating best model ..
Step 14000 of 100000; Loss: 2.4406e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5916e+03; Test Loss: 9.1948e+02. (Time: 2.7s)
first loss: 2440.560129394531 test loss new: 2438.2071240234377
updating best model ..
Step 15000 of 100000; Loss: 2.4382e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5898e+03; Test Loss: 9.1873e+02. (Time: 2.7s)
first loss: 2438.2071240234377 test loss new: 2436.044049072266
updating best model ..
Step 16000 of 100000; Loss: 2.4360e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5879e+03; Test Loss: 9.1755e+02. (Time: 2.7s)
first loss: 2436.044049072266 test loss new: 2434.451455078125
updating best model ..
Step 17000 of 100000; Loss: 2.4345e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5839e+03; Test Loss: 9.1617e+02. (Time: 2.5s)
first loss: 2434.451455078125 test loss new: 2431.5249291992186
updating best model ..
Step 18000 of 100000; Loss: 2.4315e+03. (Time: 0.0s)
Step 1000 of 1000; Los

## RL-ANN Inner-Outerr


In [ ]:
AS_OUTER_LOOP = [1,2,3,4,5,6,7,8,9,10] # assigned outer loop(s) to run in this notebook

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_rlann_inner_search_results_sub3.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_rlann_outer_results_sub3.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'False'
            use_previous_values = 'False'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.  # This is needed for it to be doing RL

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_rlann():
              model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_rlann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_rlann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'False'
    use_previous_values = 'False'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.  # This is needed for it to be doing RL

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_rlann():
      model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_rlann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_rlann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


Streaming output truncated to the last 5000 lines.
Step 15000 of 100000; Loss: 2.5076e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.7835e+03; Test Loss: 8.9507e+02. (Time: 3.1s)
first loss: 2507.5975598144532 test loss new: 2507.6440014648438

Stopping early as the loss at step 200 did not improve over step 1.
[All Batches Averaged] NLL: 0.4614
Step 1000 of 1000; Loss: 2.2611e+03; Test Loss: 1.1824e+03. (Time: 3.1s)
first loss: inf test loss new: 5957.729025878906
updating best model ..
Step 1000 of 100000; Loss: 5.9577e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 2.2016e+03; Test Loss: 1.1402e+03. (Time: 3.1s)
first loss: 5957.729025878906 test loss new: 3268.754189453125
updating best model ..
Step 2000 of 100000; Loss: 3.2688e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 2.0880e+03; Test Loss: 1.0718e+03. (Time: 3.2s)
first loss: 3268.754189453125 test loss new: 3175.829814453125
updating best model ..
Step 3000 of 100000; Loss: 3.1758e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.9563e+03

## Context-ANN inner-outer

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_conann_inner_search_results_sub3.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_conann_outer_results_sub3.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'False'
            use_previous_values = 'True'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.  # This is needed for it to be doing RL

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_conann():
              model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_conann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_conann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'False'
    use_previous_values = 'True'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.  # This is needed for it to be doing RL

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_conann():
      model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_conann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_conann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


Streaming output truncated to the last 5000 lines.
first loss: 2308.9640490722654 test loss new: 2306.5292810058595
updating best model ..
Step 6000 of 100000; Loss: 2.3065e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5421e+03; Test Loss: 7.4533e+02. (Time: 3.2s)
first loss: 2306.5292810058595 test loss new: 2304.0735668945313
updating best model ..
Step 7000 of 100000; Loss: 2.3041e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5370e+03; Test Loss: 7.4617e+02. (Time: 3.2s)
first loss: 2304.0735668945313 test loss new: 2304.3071411132814

Stopping early as the loss at step 200 did not improve over step 1.
[All Batches Averaged] NLL: 0.4054
Step 1000 of 1000; Loss: 1.7079e+03; Test Loss: 8.5037e+02. (Time: 3.2s)
first loss: inf test loss new: 3403.5115869140627
updating best model ..
Step 1000 of 100000; Loss: 3.4035e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.6159e+03; Test Loss: 7.9580e+02. (Time: 3.2s)
first loss: 3403.5115869140627 test loss new: 2286.9025537109374
updating best mode

## Memory-ANN inner-outer

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_memann_inner_search_results_sub3.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_memann_outer_results_sub3.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'True'
            use_previous_values = 'False'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.  # This is needed for it to be doing RL

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_memann():
              model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_memann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_memann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'True'
    use_previous_values = 'False'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.  # This is needed for it to be doing RL

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_memann():
      model = hybrnn.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_memann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_memann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


Streaming output truncated to the last 5000 lines.
updating best model ..
Step 4000 of 100000; Loss: 1.5175e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.7684e+03; Test Loss: 8.6108e+02. (Time: 3.6s)
first loss: 1517.5077563476561 test loss new: 1413.8833618164062
updating best model ..
Step 5000 of 100000; Loss: 1.4139e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 8.5167e+02; Test Loss: 8.7746e+02. (Time: 3.6s)
first loss: 1413.8833618164062 test loss new: 1352.0574340820312
updating best model ..
Step 6000 of 100000; Loss: 1.3521e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.7159e+03; Test Loss: 8.1995e+02. (Time: 3.5s)
first loss: 1352.0574340820312 test loss new: 1312.8958874511718
updating best model ..
Step 7000 of 100000; Loss: 1.3129e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 8.4655e+02; Test Loss: 1.4494e+03. (Time: 3.5s)
first loss: 1312.8958874511718 test loss new: 1308.2755712890626
updating best model ..
Step 8000 of 100000; Loss: 1.3083e+03. (Time: 0.0s)
Step 1000 of 1000; Los

## Mixed Gated Model

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
import jax
import jax.numpy as jnp
import haiku as hk
import optax
from sklearn.model_selection import KFold

# =========================================================
# Part 2: 完整训练脚本 (Nested CV with Gradient Clipping)
# =========================================================

# Step 0: 数据准备
# ---------------------------------------------------------
# 确保 xs_human_list, ys_human_list, rnn_utils 已经在环境上下文中
if 'xs_human_list' not in locals() or 'rnn_utils' not in locals():
    raise ValueError("请先加载数据 (xs_human_list) 和工具库 (rnn_utils)！")

xs_all = np.concatenate(xs_human_list, axis=1)
ys_all = np.concatenate(ys_human_list, axis=1)

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape
print(f"Data Loaded: {n_trials} trials, {total_sessions} sessions.")

# Step 1: Dataset Helper
# ---------------------------------------------------------
def format_into_datasets_by_index(xs, ys, dataset_constructor, train_idx, val_idx, test_idx, batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# Step 2: 超参数设置 (优化版)
# ---------------------------------------------------------
param_grid = {
    # 锁定在 8 和 16，这是 Gated RNN 的"甜点区"
    'hidden_size': [4, 8, 16, 32],

    # 【关键修改】调低学习率，配合 Gradient Clipping 使用
    # 1e-3 对于 Gate 结构来说太大了，容易导致 early collapse
    'lr': [5e-4, 1e-4],

    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# Step 3: 结果路径
# ---------------------------------------------------------
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_gated_inner_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_gated_outer_results.csv')

# 初始化 CSV (如果不存在)
if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=['outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'])

if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=['outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'])

# Step 4: 外层循环 (Outer CV)
# ---------------------------------------------------------
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

if 'AS_OUTER_LOOP' not in locals():
    AS_OUTER_LOOP = list(range(1, K_outer + 1))

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned)")
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        # 检查是否已跑过
        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config: hs={hs}, lr={lr}...")
            # 如果需要恢复 best_score 逻辑，可在这里读取 CSV
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):
            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_val_idx[inner_train_idx],
                val_idx=train_val_idx[inner_val_idx],
                test_idx=[],
                batch_size=bs
            )

            # RL & Network Params
            rnn_rl_params = {'s': True, 'o': False, 'fit_forget': False, 'forget': 0., 'w_h': 1., 'w_v': 1.}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_memann():
                return hybrnn.BiControlRNN(rl_params=rnn_rl_params, network_params=network_params)

            # --- Inner Training ---
            params, _, _ = rnn_utils.fit_model(
                model_fun=make_memann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                # 【安全锁】: 梯度裁剪 + 较低的学习率
                optimizer=optax.chain(
                    optax.clip_by_global_norm(1.0),
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=200000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_memann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)

        # Save Inner Result
        new_row = pd.DataFrame([{
            'outer_fold': outer_fold,
            'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
            'avg_val_nll': mean_inner_score,
            'time': time.time() - t0
        }])
        df_inner_existing = pd.concat([df_inner_existing, new_row], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params Fold {outer_fold}: {best_params}, Val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    rnn_rl_params = {'s': True, 'o': False, 'fit_forget': False, 'forget': 0., 'w_h': 1., 'w_v': 1.}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_memann():
      # 【修正】: 必须使用 BiControlRNN，防止用错模型
      return hybrnn.BiControlRNN(rl_params=rnn_rl_params, network_params=network_params)

    # --- Final Training ---
    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_memann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        # 【安全锁】: 梯度裁剪
        optimizer=optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=200000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_memann, params)
    print(f"Outer fold {outer_fold} Test NLL: {test_nll:.4f}")

    # Save Outer Result
    new_outer_row = pd.DataFrame([{
        'outer_fold': outer_fold,
        'best_hidden_size': best_params['hidden_size'],
        'best_lr': best_params['lr'],
        'best_weight_decay': best_params['weight_decay'],
        'best_batch_size': best_params['batch_size'],
        'test_nll': test_nll
    }])
    df_outer_existing = pd.concat([df_outer_existing, new_outer_row], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# Step 5: Summary
# ---------------------------------------------------------
if len(df_outer_existing) > 0:
    avg_nll = df_outer_existing['test_nll'].mean()
    se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
    print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")

Streaming output truncated to the last 5000 lines.
first loss: 2264.624815551758 test loss new: 2262.6055510864257
updating best model ..
Step 33000 of 200000; Loss: 2.2626e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.4408e+03; Test Loss: 7.8725e+02. (Time: 5.4s)
first loss: 2262.6055510864257 test loss new: 2260.7709102172853
updating best model ..
Step 34000 of 200000; Loss: 2.2608e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.4408e+03; Test Loss: 7.8729e+02. (Time: 5.4s)
first loss: 2260.7709102172853 test loss new: 2259.003730773926
updating best model ..
Step 35000 of 200000; Loss: 2.2590e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.4397e+03; Test Loss: 7.8732e+02. (Time: 5.4s)
first loss: 2259.003730773926 test loss new: 2257.7111385498047
updating best model ..
Step 36000 of 200000; Loss: 2.2577e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.4386e+03; Test Loss: 7.8721e+02. (Time: 5.4s)
first loss: 2257.7111385498047 test loss new: 2256.653288269043
updating best model ..
Step 3700

## Mixed Ungated Model

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
import jax
import jax.numpy as jnp
import haiku as hk
import optax
from sklearn.model_selection import KFold

# =========================================================
# Part 2: 完整训练脚本 (Nested CV with Gradient Clipping)
# =========================================================

# Step 0: 数据准备
# ---------------------------------------------------------
# 确保 xs_human_list, ys_human_list, rnn_utils 已经在环境上下文中
if 'xs_human_list' not in locals() or 'rnn_utils' not in locals():
    raise ValueError("请先加载数据 (xs_human_list) 和工具库 (rnn_utils)！")

xs_all = np.concatenate(xs_human_list, axis=1)
ys_all = np.concatenate(ys_human_list, axis=1)

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape
print(f"Data Loaded: {n_trials} trials, {total_sessions} sessions.")

# Step 1: Dataset Helper
# ---------------------------------------------------------
def format_into_datasets_by_index(xs, ys, dataset_constructor, train_idx, val_idx, test_idx, batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# Step 2: 超参数设置 (优化版)
# ---------------------------------------------------------
param_grid = {
    # 锁定在 8 和 16，这是 Gated RNN 的"甜点区"
    'hidden_size': [8, 16],

    # 【关键修改】调低学习率，配合 Gradient Clipping 使用
    # 1e-3 对于 Gate 结构来说太大了，容易导致 early collapse
    'lr': [5e-4, 1e-4],

    'weight_decay': [1e-3, 1e-4],
    'batch_size': [64],
}
K_outer = 10
K_inner = 3
random_seed = 42

# Step 3: 结果路径
# ---------------------------------------------------------
OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'human_ungated_inner_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'human_ungated_outer_results.csv')

# 初始化 CSV (如果不存在)
if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=['outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'])

if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=['outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'])

# Step 4: 外层循环 (Outer CV)
# ---------------------------------------------------------
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

if 'AS_OUTER_LOOP' not in locals():
    AS_OUTER_LOOP = list(range(1, K_outer + 1))

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned)")
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        # 检查是否已跑过
        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config: hs={hs}, lr={lr}...")
            # 如果需要恢复 best_score 逻辑，可在这里读取 CSV
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):
            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_val_idx[inner_train_idx],
                val_idx=train_val_idx[inner_val_idx],
                test_idx=[],
                batch_size=bs
            )

            # RL & Network Params
            rnn_rl_params = {'s': True, 'o': False, 'fit_forget': False, 'forget': 0., 'w_h': 1., 'w_v': 1.}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_memann():
                return hybrnn.BiAdditiveRNN(rl_params=rnn_rl_params, network_params=network_params)

            # --- Inner Training ---
            params, _, _ = rnn_utils.fit_model(
                model_fun=make_memann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                # 【安全锁】: 梯度裁剪 + 较低的学习率
                optimizer=optax.chain(
                    optax.clip_by_global_norm(1.0),
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=200000,
                early_stop_step=200,
                if_early_stop=True
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_memann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)

        # Save Inner Result
        new_row = pd.DataFrame([{
            'outer_fold': outer_fold,
            'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
            'avg_val_nll': mean_inner_score,
            'time': time.time() - t0
        }])
        df_inner_existing = pd.concat([df_inner_existing, new_row], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params Fold {outer_fold}: {best_params}, Val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    rnn_rl_params = {'s': True, 'o': False, 'fit_forget': False, 'forget': 0., 'w_h': 1., 'w_v': 1.}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_memann():
      # 【修正】: 必须使用 BiControlRNN，防止用错模型
      return hybrnn.BiAdditiveRNN(rl_params=rnn_rl_params, network_params=network_params)

    # --- Final Training ---
    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_memann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        # 【安全锁】: 梯度裁剪
        optimizer=optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=200000,
        early_stop_step=200,
        if_early_stop=True
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_memann, params)
    print(f"Outer fold {outer_fold} Test NLL: {test_nll:.4f}")

    # Save Outer Result
    new_outer_row = pd.DataFrame([{
        'outer_fold': outer_fold,
        'best_hidden_size': best_params['hidden_size'],
        'best_lr': best_params['lr'],
        'best_weight_decay': best_params['weight_decay'],
        'best_batch_size': best_params['batch_size'],
        'test_nll': test_nll
    }])
    df_outer_existing = pd.concat([df_outer_existing, new_outer_row], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# Step 5: Summary
# ---------------------------------------------------------
if len(df_outer_existing) > 0:
    avg_nll = df_outer_existing['test_nll'].mean()
    se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
    print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")

Streaming output truncated to the last 5000 lines.
Step 5000 of 200000; Loss: 2.5963e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.6110e+03; Test Loss: 8.4559e+02. (Time: 4.0s)
first loss: 2596.264860656738 test loss new: 2524.917914733887
updating best model ..
Step 6000 of 200000; Loss: 2.5249e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5777e+03; Test Loss: 8.2761e+02. (Time: 4.1s)
first loss: 2524.917914733887 test loss new: 2479.292085571289
updating best model ..
Step 7000 of 200000; Loss: 2.4793e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 1.5607e+03; Test Loss: 8.1758e+02. (Time: 4.0s)
first loss: 2479.292085571289 test loss new: 2491.3633404541015

Stopping early as the loss at step 200 did not improve over step 1.
[All Batches Averaged] NLL: 0.4429
Step 1000 of 1000; Loss: 2.8021e+03; Test Loss: 1.6937e+03. (Time: 3.9s)
first loss: inf test loss new: 9859.055063842774
updating best model ..
Step 1000 of 200000; Loss: 9.8591e+03. (Time: 0.0s)
Step 1000 of 1000; Loss: 2.2114e+03; 

## RL Outer

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import KFold

# ==============================
# Step 0: 合并人类数据
# ==============================
# 假设 xs_human_list / ys_human_list 是 list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 合并 session
ys_all = np.concatenate(ys_human_list, axis=1)

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape
print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")

# ==============================
# Step 1: 按索引切分 dataset
# ==============================
def format_into_datasets_by_index(xs, ys, dataset_constructor,
                                  train_idx, val_idx, test_idx, batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size) if len(val_idx) > 0 else None
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# ==============================
# Step 2: Outer CV 设置
# ==============================
K_outer = 10
random_seed = 42

OUTPUT_DIR = '/content/drive/MyDrive/Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
outer_csv = os.path.join(OUTPUT_DIR, 'human_cognitive_model_outer_results.csv')

if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'test_nll'
    ])

outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

# ==============================
# Step 3: Outer loop
# ==============================
for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):
    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    # 拆分数据 (认知模型不需要 validation)
    ds_train, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=32  # 固定 batch_size 或按需要调整
    )

    # 定义认知模型
    def make_cog_model():
        return bandits.Hk_PreserveConAgentQ()

    # 训练认知模型
    rl_params, _, _ = rnn_utils.fit_model(
        model_fun=make_cog_model,
        dataset_train=ds_train,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(1e-5),  # 固定超参
            optax.adam(learning_rate=1e-4)
        ),
        n_steps_per_call=500,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    # 测试集 NLL
    test_nll = compute_negative_log_likelihood(ds_test, make_cog_model, rl_params)
    print(f"Test NLL (fold {outer_fold}): {test_nll:.4f}")

    # 保存结果
    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# ==============================
# Step 4: 最终平均 NLL ± SE
# ==============================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


# Train best models

## Vanilla RNN

### Train on all data

In [ ]:
# =====================================
# Step 6: 基于 inner-outer 结果的精炼搜索 & 最终模型
# 实现需求：
# 1) 每个外层fold选 inner 的前10配置，统计跨fold出现次数，取排名前10
# 2) 统计这10个配置的参数范围，细化range
# 3) 全数据集（10%做val，其余train）上做一次grid search，记录每组表现
# 4) 保存本次grid search的最佳参数 & 最终模型权重（在全数据上重训）
# =====================================
import os, json, time, pickle, itertools
import numpy as np
import pandas as pd
from datetime import datetime
import glob


REFINE_DIR = os.path.join('/content/drive/MyDrive/Results', "human_final_refined_grid")
os.makedirs(REFINE_DIR, exist_ok=True)

# 结果文件
ranked_top_csv     = os.path.join(REFINE_DIR, "vrnn_ranked_top10_configs_from_folds.csv")
refined_search_csv = os.path.join(REFINE_DIR, "vrnn_refined_gridsearch_val_results.csv")
final_manifest     = os.path.join(REFINE_DIR, "vrnn_final_model_manifest.json")
final_params_path  = os.path.join(REFINE_DIR, "vrnn_final_model_params.pkl")

# -----------------------------
# Step 6.1: 从 inner-outer 结果中选出“跨fold前十”的候选配置
# -----------------------------
TOP_K_PER_FOLD = 10   # 每个fold取前10
FINAL_TOP_K    = 10   # 跨fold计数后，取前10

# 读取之前保存的 inner / outer CSV
# 假设这些csv都在同一个目录，比如：
DATA_DIR = "/content/drive/MyDrive/Results"
# 匹配所有 human_vrnn_inner_search_results_sub*.csv
inner_files = glob.glob(os.path.join(DATA_DIR, "human_vrnn_inner_search_results_sub*.csv"))
print("找到的inner文件：", inner_files)
# 读入并合并
df_inner_list = [pd.read_csv(f) for f in inner_files]
df_inner = pd.concat(df_inner_list, ignore_index=True)
print("合并后的 df_inner 维度：", df_inner.shape)
print(df_inner.head())

# 以每个 outer_fold 的 inner 搜索为单位，取 avg_val_nll 最小的前TOP_K_PER_FOLD 组合
need_cols = ['outer_fold','hidden_size','lr','weight_decay','batch_size','avg_val_nll']
missing_cols = [c for c in need_cols if c not in df_inner.columns]
if missing_cols:
    raise ValueError(f"inner_csv 缺少必要列: {missing_cols}")

df_inner_topk = (df_inner[need_cols]
                 .sort_values(['outer_fold','avg_val_nll'], ascending=[True, True])
                 .groupby('outer_fold')
                 .head(TOP_K_PER_FOLD)
                 .copy())

# 将超参组合作为key，统计跨fold出现次数，并计算其在被选fold中的平均/中位验证NLL用于并列打破
df_inner_topk['key'] = list(zip(df_inner_topk['hidden_size'],
                                df_inner_topk['lr'],
                                df_inner_topk['weight_decay'],
                                df_inner_topk['batch_size']))

count_series = df_inner_topk.groupby('key').size().rename('count')
mean_val     = df_inner_topk.groupby('key')['avg_val_nll'].mean().rename('mean_val_nll')
median_val   = df_inner_topk.groupby('key')['avg_val_nll'].median().rename('median_val_nll')

rank_table = (pd.concat([count_series, mean_val, median_val], axis=1)
                .reset_index()
                .sort_values(['count','mean_val_nll','median_val_nll'],
                             ascending=[False, True, True])
                .head(FINAL_TOP_K))

# 展开 key
rank_table[['hidden_size','lr','weight_decay','batch_size']] = pd.DataFrame(rank_table['key'].tolist(),
                                                                           index=rank_table.index)
rank_table = rank_table[['hidden_size','lr','weight_decay','batch_size','count','mean_val_nll','median_val_nll']]
rank_table.to_csv(ranked_top_csv, index=False)
print(f"[Step6.1] 跨fold计数后Top-{FINAL_TOP_K}配置已保存 -> {ranked_top_csv}")

# # -----------------------------
# # Step 6.2: 统计这十个配置的范围，并细化range
# # - hidden_size / batch_size: 使用出现过的唯一值集合（保持离散）
# # -----------------------------
# top10 = rank_table.copy()
# hs_values = sorted(set(int(v) for v in top10['hidden_size'].tolist()))
# bs_values = sorted(set(int(v) for v in top10['batch_size'].tolist()))
# lr_values = sorted(set(v for v in top10['lr'].tolist()))
# wd_values = sorted(set(v for v in top10['weight_decay'].tolist()))

# print(f"[Step6.2] 精炼搜索空间：")
# print(f"  hidden_size: {hs_values}")
# print(f"  batch_size : {bs_values}")
# print(f"  lr         : {lr_values}")
# print(f"  weight_decay: {wd_values}")

import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
# param_grid = {
#     'hidden_size': hs_values,
#     'lr': lr_values,
#     'weight_decay': wd_values,
#     'batch_size': bs_values,
# }

param_grid = {
    'hidden_size': [8, 16, 32, 64],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [32, 64],
}

random_seed = 42
rng = np.random.default_rng(random_seed)
session_indices = np.arange(total_sessions)
n_val = max(1, int(0.10 * total_sessions))
val_sessions = rng.choice(session_indices, size=n_val, replace=False)
train_sessions = np.array([s for s in session_indices if s not in set(val_sessions)])

print(f"[Split] Train sessions: {len(train_sessions)} | Val sessions: {len(val_sessions)}")

# -------- Grid Search 主循环（带早停），增量写盘 --------
rows = []
done_keys = set()
if os.path.exists(refined_search_csv):
    # 允许断点续跑
    df_old = pd.read_csv(refined_search_csv)
    rows = df_old.to_dict('records')
    done_keys = set((int(r['hidden_size']), float(r['lr']), float(r['weight_decay']), int(r['batch_size'])) for r in rows)

t0 = time.time()
best_score = float('inf')
best_rec = None

for hs, lr, wd, bs in itertools.product(
        param_grid['hidden_size'],
        param_grid['lr'],
        param_grid['weight_decay'],
        param_grid['batch_size']):

    key = (int(hs), float(lr), float(wd), int(bs))
    if key in done_keys:
        print(f"[Skip] Done: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
        continue

    ds_train, ds_val, _ = format_into_datasets_by_index(
        xs_all, ys_all, rnn_utils.DatasetRNN,
        train_idx=train_sessions, val_idx=val_sessions, test_idx=[],
        batch_size=bs
    )

    def make_rnn():
        return hk.DeepRNN([hk.VanillaRNN(hs), hk.Linear(output_size=2)])

    params, _, best_step = rnn_utils.fit_model(
        model_fun=make_rnn,
        dataset_train=ds_train,
        dataset_test=ds_val,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(wd),
            optax.adam(learning_rate=lr)
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )

    val_nll = evaluate_nll_over_full_dataset(ds_val, make_rnn, params)

    rec = {
        'hidden_size': int(hs),
        'lr': float(lr),
        'weight_decay': float(wd),
        'batch_size': int(bs),
        'val_nll': float(val_nll),
        'best_step': int(best_step) if best_step is not None else None,
        'time_sec': time.time() - t0
    }
    rows.append(rec)
    done_keys.add(key)

    # 增量保存（断点安全）
    pd.DataFrame(rows).to_csv(refined_search_csv, index=False)
    print(f"[Grid] hs={hs}, bs={bs}, lr={lr:.2e}, wd={wd:.2e} -> val NLL={val_nll:.4f}, best_step={best_step}")

    if val_nll < best_score:
        best_score = val_nll
        best_rec = rec

# 汇总排序一次
df_refine = pd.DataFrame(rows).sort_values('val_nll', ascending=True, kind='mergesort')
df_refine.to_csv(refined_search_csv, index=False)
print(f"\n[Grid] 完成，共评估 {len(df_refine)} 组；最佳验证 NLL={df_refine.iloc[0]['val_nll']:.6f}")

import pandas as pd
# -------- 选最佳组合并在“全数据”上重训（固定步数）--------
best_cfg = df_refine.iloc[0].to_dict()
best_hs = int(best_cfg['hidden_size'])
best_bs = int(best_cfg['batch_size'])
best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
fixed_steps = int(best_cfg['best_step']) if pd.notnull(best_cfg['best_step']) else 20000  # 兜底

print(f"[Final] Best: hs={best_hs}, bs={best_bs}, lr={best_lr:.2e}, wd={best_wd:.2e}, steps={fixed_steps}")

# 全数据（train+val合并）作为训练集
ds_train, _, _ = format_into_datasets_by_index(
        xs_all, ys_all, rnn_utils.DatasetRNN,
        train_idx=np.arange(xs_all.shape[1]), val_idx=[], test_idx=[],
        batch_size=best_bs
    )
def make_rnn():
    return hk.DeepRNN([hk.VanillaRNN(best_hs), hk.Linear(output_size=2)])

final_params, _, _ = rnn_utils.fit_model(
    model_fun=make_rnn,
    dataset_train=ds_train,
    dataset_test=None,           # 不再用验证集
    loss_fun='categorical',
    optimizer=optax.chain(
        optax.add_decayed_weights(best_wd),
        optax.adam(learning_rate=best_lr)
    ),
    n_steps_per_call=1000,
    n_steps_max=500000,
    early_stop_step=200,
    if_early_stop=False
)

# 仅作记录（有偏）的全数据 NLL
ref_nll = evaluate_nll_over_full_dataset(ds_train, make_rnn, final_params)
print(f"[Final] Reference NLL on ALL data (biased) = {ref_nll:.6f}")

# -------- 保存权重与manifest --------
with open(final_params_path, "wb") as f:
    pickle.dump(final_params, f)

manifest = {
    "random_seed": random_seed,
    "split": {"val_fraction": 0.10, "n_sessions": int(total_sessions)},
    "grid_space": {k: list(map(lambda x: float(x) if isinstance(x, (np.floating,np.float32,np.float64)) else int(x) if isinstance(x, (np.integer,np.int32,np.int64)) else x, v))
                   for k, v in param_grid.items()},
    "best_config": {
        "hidden_size": best_hs,
        "batch_size": best_bs,
        "lr": best_lr,
        "weight_decay": best_wd
    },
    "fixed_train_steps": fixed_steps,
    "reference_full_data_nll": float(ref_nll),
    "results_csv": refined_search_csv,
    "params_path": final_params_path
}
with open(final_manifest, "w") as f:
    json.dump(manifest, f, indent=2)

print("\n[Outputs]")
print("  Grid results CSV :", refined_search_csv)
print("  Final manifest   :", final_manifest)
print("  Final params     :", final_params_path)


### Best model for each outer fold

In [ ]:
# =====================================
# Step 6: 基于 inner-outer 结果的精炼搜索 & 最终模型
# 这里将 Step 6.1 改为：为每个 outer_fold 选出单一最佳参数，并保存到 CSV
# =====================================
import os, json, time, pickle, itertools
import numpy as np
import pandas as pd
from datetime import datetime
import glob

REFINE_DIR = os.path.join('/content/drive/MyDrive/Results', "human_final_refined_grid")
os.makedirs(REFINE_DIR, exist_ok=True)

# 输出文件
ranked_top_csv           = os.path.join(REFINE_DIR, "vrnn_ranked_top10_configs_from_folds.csv")  # 若你下游还需要这个名字，可保留，但此处不再写入
best_per_fold_csv        = os.path.join(REFINE_DIR, "vrnn_best_params_per_fold.csv")              # ★ 新增：每个fold的最佳参数（单组）
refined_search_csv       = os.path.join(REFINE_DIR, "vrnn_refined_gridsearch_val_results.csv")
final_manifest           = os.path.join(REFINE_DIR, "vrnn_final_model_manifest.json")
final_params_path        = os.path.join(REFINE_DIR, "vrnn_final_model_params.pkl")

# -----------------------------
# Step 6.1: 为每个 outer_fold 选出“单一最佳参数”并保存
# -----------------------------
DATA_DIR = "/content/drive/MyDrive/Results"
inner_files = glob.glob(os.path.join(DATA_DIR, "human_vrnn_inner_search_results_sub*.csv"))
print("找到的 inner 文件：", inner_files)

if not inner_files:
    raise FileNotFoundError("未找到 human_vrnn_inner_search_results_sub*.csv，请检查路径与文件名。")

# 合并 inner 结果
df_inner_list = [pd.read_csv(f) for f in inner_files]
df_inner = pd.concat(df_inner_list, ignore_index=True)
print("合并后的 df_inner 维度：", df_inner.shape)
print(df_inner.head())

# 必要列校验
need_cols = ['outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll']
missing_cols = [c for c in need_cols if c not in df_inner.columns]
if missing_cols:
    raise ValueError(f"inner_csv 缺少必要列: {missing_cols}")

# 为稳妥起见，确保数值列类型正确
df_inner['outer_fold']    = df_inner['outer_fold'].astype(int)
df_inner['hidden_size']   = df_inner['hidden_size'].astype(int)
df_inner['batch_size']    = df_inner['batch_size'].astype(int)
df_inner['lr']            = df_inner['lr'].astype(float)
df_inner['weight_decay']  = df_inner['weight_decay'].astype(float)
df_inner['avg_val_nll']   = df_inner['avg_val_nll'].astype(float)

# 方式一：按 outer_fold 分组，在组内按 avg_val_nll 升序；并列时再用其它列稳定打破，取每组首行
df_best_per_fold = (
    df_inner[need_cols]
    .sort_values(
        ['outer_fold', 'avg_val_nll', 'hidden_size', 'lr', 'weight_decay', 'batch_size'],
        ascending=[True, True, True, True, True, True]
    )
    .groupby('outer_fold', as_index=False)
    .first()
)

# （可选）方式二：用 idxmin 精确选出每组最小 avg_val_nll 的行
# idx = df_inner.groupby('outer_fold')['avg_val_nll'].idxmin()
# df_best_per_fold = df_inner.loc[idx, need_cols].sort_values('outer_fold').reset_index(drop=True)

# 保存为“每个fold一组最佳参数”的 CSV
df_best_per_fold.to_csv(best_per_fold_csv, index=False)
print(f"[Step6.1] 每个 outer_fold 的最佳参数已保存 -> {best_per_fold_csv}")
print(df_best_per_fold)




# ============================================================
# Outer-loop: 为每个 fold 从 ranked_top10 CSV 里挑最佳参数训练 & 保存
#  - 训练集：train_val（合并）；验证/评估：test
#  - 候选参数来源：vrnn_ranked_top10_configs_from_folds.csv（ranked_top_csv）
#  - 输出：
#      {REFINE_DIR}/folds/fold{K}/vrnn_final_model_fold.pkl
#      {REFINE_DIR}/folds/fold{K}/vrnn_final_model_manifest.json
#      {REFINE_DIR}/vrnn_final_folds_summary.csv
# ============================================================
import os, json, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# --- 参数与路径 ---
K_outer = 10  # 与你外层CV保持一致
FINAL_FOLDS_DIR = os.path.join(REFINE_DIR, "folds")
os.makedirs(FINAL_FOLDS_DIR, exist_ok=True)
final_folds_csv = os.path.join(FINAL_FOLDS_DIR, "vrnn_final_folds_summary.csv")
random_seed=42

# # --- 读取候选参数（Top-10 跨fold计数后的列表）---
# if not os.path.exists(ranked_top_csv):
#     raise FileNotFoundError(f"未找到候选参数CSV: {ranked_top_csv}")

df_cands = pd.read_csv(best_per_fold_csv)
need_cols = {'hidden_size','lr','weight_decay','batch_size'}
if not need_cols.issubset(df_cands.columns):
    raise ValueError(f"{ranked_top_csv} 缺少列: {need_cols - set(df_cands.columns)}")

# 规范类型，避免科学计数法被读成字符串
df_cands['hidden_size']  = df_cands['hidden_size'].astype(int)
df_cands['batch_size']   = df_cands['batch_size'].astype(int)
df_cands['lr']           = df_cands['lr'].astype(float)
df_cands['weight_decay'] = df_cands['weight_decay'].astype(float)

# 转为候选列表（按你CSV顺序即可）
candidate_configs = df_cands[['hidden_size','lr','weight_decay','batch_size']].to_dict('records')

# --- 外层折分（与外层代码一致）---
session_indices = np.arange(xs_all.shape[1])
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)

def _make_rnn(hs):
    return hk.DeepRNN([hk.VanillaRNN(int(hs)), hk.Linear(output_size=2)])

def _train_and_eval_on_split(cfg, train_val_idx, test_idx):
    """在 train_val 上训练、test 上做早停与评估，返回 (params, best_step, test_nll)。"""
    hs = int(cfg['hidden_size']); bs = int(cfg['batch_size'])
    lr = float(cfg['lr']);        wd = float(cfg['weight_decay'])

    ds_train, ds_test, _ = format_into_datasets_by_index(
        xs_all, ys_all, rnn_utils.DatasetRNN,
        train_idx=train_val_idx,  # 训练：train∪val
        val_idx=test_idx,         # 这里占位未用
        test_idx=[],              # 不额外构造第三个集
        batch_size=bs
    )
    # 注意：把“测试集”作为 fit 的 dataset_test，用于早停/监控
    params, _, best_step = rnn_utils.fit_model(
        model_fun=lambda: _make_rnn(hs),
        dataset_train=ds_train,
        dataset_test=ds_test,   # 用 test 做监控与早停（按你的需求）
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(wd),
            optax.adam(learning_rate=lr)
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True
    )
    test_nll = evaluate_nll_over_full_dataset(ds_test, lambda: _make_rnn(hs), params)
    return params, best_step, float(test_nll)

summaries = []
for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):
    print(f"\n=== Per-fold final training: Outer Fold {outer_fold}/{K_outer} ===")
    # 在该 fold 上遍历候选参数，选 test NLL 最小的
    best = {"nll": np.inf, "cfg": None, "params": None, "best_step": None}
    cfg = candidate_configs[outer_fold-1]
    params, best_step, nll = _train_and_eval_on_split(cfg, train_val_idx, test_idx)
    print(f"[Fold {outer_fold}] hs={cfg['hidden_size']}, bs={cfg['batch_size']}, "
           f"lr={cfg['lr']:.2e}, wd={cfg['weight_decay']:.2e} -> test NLL={nll:.6f}, best_step={best_step}")
    if nll < best["nll"]:
        best.update({"nll": nll, "cfg": cfg, "params": params, "best_step": best_step})

    # 保存该 fold 的最佳模型
    # fold_dir = os.path.join(FINAL_FOLDS_DIR, f"fold{outer_fold}")
    # os.makedirs(fold_dir, exist_ok=True)
    params_path = os.path.join(FINAL_FOLDS_DIR, f"vrnn_final_model_fold{outer_fold}.pkl")
    with open(params_path, "wb") as f:
        pickle.dump(best["params"], f)

    # 写 manifest
    manifest = {
        "outer_fold": int(outer_fold),
        "train_val_idx": list(map(int, train_val_idx)),
        "test_idx": list(map(int, test_idx)),
        "best_config": {
            "hidden_size": int(best["cfg"]['hidden_size']),
            "batch_size": int(best["cfg"]['batch_size']),
            "lr": float(best["cfg"]['lr']),
            "weight_decay": float(best["cfg"]['weight_decay']),
        },
        "best_step": int(best["best_step"]) if best["best_step"] is not None else None,
        "test_nll": float(best["nll"]),
        "params_path": params_path
    }
    with open(os.path.join(FINAL_FOLDS_DIR, "vrnn_final_model_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    summaries.append(manifest)
    print(f"[Saved] Fold {outer_fold}: {params_path} (test NLL={best['nll']:.6f})")

# 汇总 CSV
if summaries:
    df_sum = pd.DataFrame(summaries).sort_values("outer_fold")
    df_sum.to_csv(final_folds_csv, index=False)
    print("\n[Per-fold Final] Summary saved ->", final_folds_csv)


## RL Model

### Train on all data

### Best model for each outer fold